# Agent AI Bootcamp

Code from the section notes goes into cells below. Keys are read from `.env` in the project root.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

# Groq (OpenAI-compatible). To use OpenAI instead, replace with: openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

Quick API check. Run this after putting your real Groq key in `.env`; a 401 error means the key is wrong or still the placeholder.

In [2]:
response = openai_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Reply with the single word: ok"}],
)
print(response.choices[0].message.content)

ok


## 03 RAG: Retrieval-Augmented Generation

Course notes use OpenAI `responses.create` with `gpt-5.4-mini`. We use Groq, so the same idea is written with `chat.completions` and `openai/gpt-oss-120b`.

In [3]:
MODEL = "openai/gpt-oss-120b"

def llm(prompt):
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [4]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. How’s your day going?'

### Plain LLM lacks our data
A course-specific question gets a generic answer: the model knows nothing about Zoomcamp enrollment.

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely! You can enroll in the course at any time—there’s no deadline for signing up. Here’s a quick guide to get you started:

1. **Visit the Course Page**  
   Go to the course’s homepage (or the learning platform you’re using) and click the “Enroll” or “Join Now” button.

2. **Create an Account / Log In**  
   If you don’t already have an account, you’ll need to register with your name, email, and a password. If you already have an account, just log in.

3. **Select Your Plan**  
   - **Free trial / audit:** Many courses let you preview the material for free.  
   - **Paid enrollment:** If the course has a fee, you can choose a payment option (credit card, PayPal, etc.) and complete the checkout.

4. **Confirm Your Enrollment**  
   After payment (or choosing the free option), you’ll receive a confirmation email with a link to the course dashboard.

5. **Start Learning**  
   - **Orientation:** Some courses include an introductory video or welcome module—great for getting familia

### Adding context manually
Paste FAQ entries into the prompt. This is RAG done by hand.

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes—you can still join the course. Just keep in mind that if you’d like to receive a certificate, you’ll need to submit your final project before the submission deadline closes.


### RAG = search + prompt + LLM

```
User question -> search(knowledge base) -> top documents -> build_prompt(question, docs) -> llm(prompt) -> answer
```

The LLM only sees the documents we hand it, so answers are grounded in our data. Retrieval quality decides answer quality. `search` and `build_prompt` are implemented in the next sections (dataset + minsearch); the skeleton below is the whole architecture.

In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

# search() and build_prompt() come in the next section

## 04 The Course FAQ Dataset
FAQ data from DataTalks.Club as JSON. This is our RAG knowledge base.

In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
len(courses_raw), courses_raw[0]

(6,
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 482})

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1456

In [12]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [13]:
from collections import Counter
Counter(d["course"] for d in documents)

Counter({'machine-learning-zoomcamp': 482,
         'data-engineering-zoomcamp': 398,
         'mlops-zoomcamp': 249,
         'llm-zoomcamp': 161,
         'stock-markets-analytics-zoomcamp': 93,
         'ai-dev-tools-zoomcamp': 73})

## 05 Search with minsearch
Text fields are tokenized and ranked; keyword fields are exact-match filters.

**Windows note:** real `minsearch` fails to import here (Smart App Control blocks scipy `_ckdtree` DLL). The cell falls back to `minsearch_lite.py`, same API, standard library only.

In [14]:
# Course code: from minsearch import Index
# Smart App Control blocks scipy DLLs that minsearch needs (via scikit-learn),
# so we use a pure-Python drop-in with the same API: notebooks/minsearch_lite.py
try:
    from minsearch import Index
except ImportError:
    from minsearch_lite import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

### Boosting fields

In [16]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)
[(doc["course"], doc["question"]) for doc in results]

[('llm-zoomcamp', 'I just discovered the course. Can I still join?'),
 ('machine-learning-zoomcamp',
  'The course has already started. Can I still join it?'),
 ('mlops-zoomcamp',
  'Course - Can I still join the course after the start date?'),
 ('data-engineering-zoomcamp',
  'Course: Can I still join the course after the start date?'),
 ('data-engineering-zoomcamp', 'Course: When does the course start?')]

### Filtering by course

In [17]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Course: How do I start?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?']

### search() function: first RAG component

In [18]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_results = search(question)
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

### Lexical search limitation
Same meaning, different words. Text search only matches words.

In [19]:
[doc["question"] for doc in search("Is it possible to enroll late?")]

['Is it a group project?',
 "My dataset is copyrighted and I can't publish it. Can I still use it for the project?",
 'uv says Failed to hardlink files in Codespaces. Is it an error?',
 'My knowledge base file is too large for the repository. How do I share it with reviewers?',
 'What happens to code saved in Codespaces if I do not commit it?']

## 06 Building the Prompt
Instructions (system prompt) stay fixed. The user prompt changes per request: question + retrieved context.

In [20]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [21]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [22]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [23]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [24]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w